# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list out the record sets, fields, and columns referenced by their `@id`. This helps identify which components are available for extraction and processing.

In [ ]:
# Explore available record sets and fields
record_sets = []
for rs in metadata.recordSet:
    print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    record_sets.append(rs['@id'])
    # List fields in this record set
    if 'field' in rs:
        for field in rs['field']:
            print(f"   Field @id: {field['@id']} | Name: {field.get('name', 'N/A')} | Data type: {field.get('dataType', 'N/A')}")
            if 'column' in field:
                for column in field['column']:
                    print(f"      Column @id: {column['@id']} | Name: {column.get('name', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All access is via `@id` of entities.

In [ ]:
# Extract data from record sets found above
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} loaded with columns:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering and normalization.

Let's select one record set and a numeric field for EDA. All references will use `@id`. We'll filter records, normalize the numeric column, and optionally group by another field.

In [ ]:
# Example: choose the first record set and a numeric field for EDA
if len(record_sets) == 0:
    raise Exception("No record sets found in metadata")

# Use the first record set for demonstration
record_set_id = record_sets[0]
df = dataframes[record_set_id]

# List columns to pick a numeric field by @id
print("Columns available:", df.columns.tolist())

# Suppose the dataset has a column named 'cr:coveragePercentage' (replace with your actual @id)
numeric_field_id = 'cr:coveragePercentage' if 'cr:coveragePercentage' in df.columns else df.columns[0]
group_field_id = 'cr:description' if 'cr:description' in df.columns else None

# Filtering numeric values > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping if group field exists
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships by field `@id`. We plot the normalized numeric field and, if available, boxplot/grouped plots.

In [ ]:
# Basic histogram and boxplot for numeric fields
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(filtered_df[numeric_field_id])
plt.title(f"Boxplot of {numeric_field_id} (filtered)")
plt.xlabel(numeric_field_id)
plt.show()

# If group_field_id exists, plot grouped means
if group_field_id is not None and group_field_id in filtered_df.columns:
    plt.figure(figsize=(12,6))
    sns.barplot(x=grouped_df.index, y=grouped_df[numeric_field_id])
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and process a Croissant dataset using `mlcroissant`. We explored the record sets and fields by `@id`, filtered and normalized a numeric field, visualized distributions, and optionally grouped by another key field. These steps can be extended to further analysis or downstream machine learning tasks.
